# Boom-Baam: Satellite & Aerial Classification Pipeline (Memory Optimized)

This notebook implements a **Satellite-Optimized** classification pipeline. It handles the specific challenges of blurry aerial imagery and is optimized for Kaggle's GPU memory limits.

### Satellite Enhancements:
- **Contrast Sharpening**: Helps the model see features through atmospheric blur.
- **Memory Optimization**: Adjusted Batch Size (16) and Resolution (160) to prevent OOM errors.
- **High-Capacity Backbone**: ResNet101V2 fine-tuned for blurred textures.
- **Augmentation**: 360-degree rotations and dual-axis flips.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, applications, callbacks
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from PIL import Image

# --- MEMORY-SAFE CONFIGURATION ---
BASE_PATH = "/kaggle/input/datasets/purvanshj009/boom-baam-dataset/"
if not os.path.exists(BASE_PATH):
    BASE_PATH = "./"

TRAIN_CSV = os.path.join(BASE_PATH, "TRAIN.csv")
TEST_CSV = os.path.join(BASE_PATH, "TEST.csv")
IMG_SIZE = (160, 160) # Balanced resolution to prevent memory crashes
BATCH_SIZE = 16 # Lowered to prevent Out-of-Memory (OOM)
EPOCHS = 20

print(f"Using Adjusted Config: {IMG_SIZE}, batch={BATCH_SIZE}")

## 1. Visual Inspection (EDA)
Reviewing sample images with labels.

In [ ]:
df = pd.read_csv(TRAIN_CSV)
plt.figure(figsize=(15, 6))
sample_df = df.sample(10).reset_index(drop=True)
for i in range(10):
    img_path = os.path.join(BASE_PATH, sample_df['IMAGE'][i])
    img = Image.open(img_path)
    plt.subplot(2, 5, i+1)
    plt.imshow(img)
    plt.title(f"Label: {sample_df['LABEL'][i]}")
    plt.axis('off')
plt.tight_layout()
plt.show()

## 2. Satellite Optimized Data Pipeline
Contrast Enhancement and Pre-fetching for high performance.

In [ ]:
df['ABS_PATH'] = df['IMAGE'].apply(lambda x: os.path.join(BASE_PATH, x))
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['LABEL'])

def load_and_preprocess(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.image.per_image_standardization(img)
    img = tf.image.adjust_contrast(img, 1.5)
    return img, label

train_ds = tf.data.Dataset.from_tensor_slices((train_df['ABS_PATH'], train_df['LABEL']))
train_ds = train_ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((val_df['ABS_PATH'], val_df['LABEL']))
val_ds = val_ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

## 3. Architecture (ResNet101V2)
Using a deep model adapted for satellite textures.

In [ ]:
def build_satellite_model():
    base = applications.ResNet101V2(input_shape=(*IMG_SIZE, 3), include_top=False, weights='imagenet')
    base.trainable = True

    data_aug = models.Sequential([
        layers.RandomFlip("both"),
        layers.RandomRotation(factor=1.0), # 360-degree rotation
        layers.RandomContrast(0.2)
    ])

    model = models.Sequential([
        data_aug,
        base,
        layers.GlobalAveragePooling2D(),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(512, activation='relu'),
        layers.Dense(15, activation='softmax')
    ])
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_satellite_model()
print("Model build complete.")

## 4. Training Loop
Now running with memory-safe batch size.

In [ ]:
early_stop = callbacks.EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True)
lr_reduce = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)

print("Starting Memory-Optimized Training Loop...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stop, lr_reduce]
)

## 5. Evaluation & Final Submission
Creating the Confusion Matrix and FINAL.csv results.

In [ ]:
# Evaluation
y_val_pred = np.argmax(model.predict(val_ds), axis=1)
y_val_true = val_df['LABEL'].values
cm = confusion_matrix(y_val_true, y_val_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d')
plt.show()

# Final Output
test_df = pd.read_csv(TEST_CSV)
test_df['ABS_PATH'] = test_df['IMAGE'].apply(lambda x: os.path.join(BASE_PATH, x))

def load_test(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.image.per_image_standardization(img)
    img = tf.image.adjust_contrast(img, 1.5)
    return img

test_ds = tf.data.Dataset.from_tensor_slices(test_df['ABS_PATH']).map(load_test).batch(BATCH_SIZE)
test_preds = np.argmax(model.predict(test_ds), axis=1)

final_df = pd.DataFrame({'IMAGE': test_df['IMAGE'], 'LABEL': test_preds})
final_df.to_csv("FINAL.csv", index=False)
print("✅ FINAL.csv created successfully.")